# ETTh эксперименты: Informer + Optuna

В этом ноутбуке запускаются эксперименты **Informer + Optuna** на датасетах `ETTh1` и `ETTh2`.

Основные шаги:

1. Настройка окружения и конфигурации экспериментов.
2. Для каждого значения `MAX_ROWS` создаётся усечённый CSV-файл.
3. Запускается Informer + Optuna для каждой комбинации датасет × `MAX_ROWS`.
4. Загружаются сохранённые предсказания Informer и строятся графики.
5. Формируется сводная таблица метрик по всем запускам.

Внешний код Informer предоставляет предсказания только для отложенной выборки, поэтому:
- для train-части строится график фактических значений;
- для валидационной/тестовой части строится график фактических и предсказанных значений.

In [ ]:
from __future__ import annotations

import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict
from typing import List
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

CURRENT_DIR: Path = Path.cwd().resolve()
REPO_ROOT_CANDIDATES: List[Path] = [
    CURRENT_DIR,
    CURRENT_DIR.parent,
]

REPO_ROOT: Path | None = None
for candidate in REPO_ROOT_CANDIDATES:
    if (candidate / 'src' / 'edlm_search').is_dir():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
            f'Cannot locate project root with "src/edlm_search" directory from "{CURRENT_DIR}".'
    )

SRC_DIR: Path = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from edlm_search.experiments import ExperimentResult, run_informer_optuna_etth_experiment
from edlm_search.experiments.datasets import load_ett_csv_dataset

In [ ]:


ENV_PATH: Path = REPO_ROOT / '.env'
if ENV_PATH.is_file():
    load_dotenv(dotenv_path=ENV_PATH)

ETT_DATA_DIR: Path = SRC_DIR / 'ETDataset' / 'ETT-small'
ETTH1_PATH: Path = ETT_DATA_DIR / 'ETTh1.csv'
ETTH2_PATH: Path = ETT_DATA_DIR / 'ETTh2.csv'

DATASET_NAMES: List[str] = ['ETTh1', 'ETTh2']

MAX_ROWS_VALUES: List[int] = [5000, 10000, 20000]

TRAIN_RATIO: float = float(os.getenv('TRAIN_RATIO', '0.8'))
TARGET_COLUMN: str = os.getenv('TARGET_COLUMN', 'OT')

INFORMER_TIMEOUT_SECONDS: int = int(os.getenv('INFORMER_TIMEOUT_SECONDS', '3600'))
INFORMER_N_TRIALS: int = int(os.getenv('INFORMER_N_TRIALS', '10'))
INFORMER_METRICS_ROOT: Path = (
        REPO_ROOT / os.getenv('INFORMER_METRICS_ROOT', 'artifacts/informer_optuna').strip()
).resolve()

INFORMER_DATASETS_ROOT: Path = (
        REPO_ROOT / 'artifacts' / 'informer_datasets'
).resolve()

PLOTS_MAX_POINTS: int = 500

LOG_LEVEL: int = logging.INFO
logging.basicConfig(
        level=LOG_LEVEL,
        format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
)

DEVICE_TYPE = os.getenv('DEVICE_TYPE', 'auto')

LOGGER = logging.getLogger('etth_informer_optuna_experiments')

INFORMER_METRICS_ROOT.mkdir(parents=True, exist_ok=True)
INFORMER_DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

LOGGER.info(f'Repository root resolved to "{REPO_ROOT}".')
LOGGER.info(f'ETT data directory resolved to "{ETT_DATA_DIR}".')
LOGGER.info(f'Informer metrics root resolved to "{INFORMER_METRICS_ROOT}".')
LOGGER.info(f'Informer datasets root resolved to "{INFORMER_DATASETS_ROOT}".')
LOGGER.info(f'Informer Optuna experiments will use device_type="{DEVICE_TYPE}".')

## Конфигурация и структуры данных

В этом разделе определяются:

- описания датасетов;
- конфигурация экспериментов Informer + Optuna;
- служебные структуры для идентификации запусков и путей к данным.

In [ ]:
@dataclass(frozen=True)
class DatasetConfig:
    """Dataset configuration for ETTh experiments."""

    name: str
    csv_path: Path
    train_ratio: float


@dataclass(frozen=True)
class InformerOptunaExperimentConfig:
    """Configuration for Informer + Optuna experiments."""

    timeout_seconds: int
    n_trials: int
    metrics_root_dir: Path
    device_type: str
    max_rows_values: List[int]
    plots_max_points: int


@dataclass(frozen=True)
class InformerRunKey:
    """Identifier of a single Informer experiment run."""

    dataset_name: str
    max_rows: int

## Вспомогательные функции и класс запуска экспериментов Informer

Далее определены функции для:

- подготовки усечённых CSV-файлов под разные `MAX_ROWS`;
- загрузки и разбиения датасетов;
- построения графиков для train и для предсказаний Informer.

Класс `InformerOptunaExperimentRunner` управляет запуском всех экспериментов и
формированием сводной таблицы метрик.

In [ ]:
def prepare_trimmed_csv(
        original_csv_path: Path,
        dataset_name: str,
        max_rows: int,
        datasets_root: Path,
) -> Path:
    """
    Create a trimmed CSV file with at most `max_rows` rows for a dataset.

    Parameters
    ----------
    original_csv_path : Path
        Path to the original full CSV file.
    dataset_name : str
        Name of the dataset.
    max_rows : int
        Maximum number of rows to keep.
    datasets_root : Path
        Root directory where trimmed CSV files will be stored.

    Returns
    -------
    Path
        Path to the trimmed CSV file.
    """
    if max_rows <= 0:
        raise ValueError('max_rows must be a positive integer.')
    if not original_csv_path.is_file():
        raise FileNotFoundError(
                f'Original CSV for dataset "{dataset_name}" not found at "{original_csv_path}".'
        )

    dataset_dir = datasets_root / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)
    trimmed_csv_path = dataset_dir / f'{dataset_name}_max_rows_{max_rows}.csv'

    df = pd.read_csv(original_csv_path)
    if len(df) > max_rows:
        df = df.head(max_rows)
    df.to_csv(trimmed_csv_path, index=False)

    LOGGER.info(
            f'Trimmed CSV for dataset="{dataset_name}" with max_rows={max_rows} '
            f'written to "{trimmed_csv_path}".'
    )
    return trimmed_csv_path


def load_train_valid_for_visualization(
        csv_path: Path,
        max_rows: int,
        train_ratio: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load and split ETTh dataset for visualization.

    Parameters
    ----------
    csv_path : Path
        Path to the CSV file.
    max_rows : int
        Maximum number of rows to load.
    train_ratio : float
        Train split ratio.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        Split (train_df, valid_df).
    """
    train_df, valid_df = load_ett_csv_dataset(
            csv_path=str(csv_path),
            max_rows=max_rows,
            train_ratio=train_ratio,
    )
    return train_df, valid_df


def plot_time_series_single(
        y_values: np.ndarray,
        title: str,
        max_points: int,
        label: str,
) -> None:
    """
    Plot a single time series.

    Parameters
    ----------
    y_values : np.ndarray
        Values to plot.
    title : str
        Title of the plot.
    max_points : int
        Maximum number of points to display.
    label : str
        Label for the series.
    """
    if max_points <= 0:
        raise ValueError('max_points must be positive.')
    length = min(len(y_values), max_points)
    index = np.arange(length)
    plt.figure(figsize=(10, 4))
    plt.plot(index, y_values[:length], label=label)
    plt.xlabel('time index')
    plt.ylabel('value')
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_true_vs_pred(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        title: str,
        max_points: int,
) -> None:
    """
    Plot ground truth and predictions for Informer evaluation subset.

    Parameters
    ----------
    y_true : np.ndarray
        Ground truth values.
    y_pred : np.ndarray
        Predicted values.
    title : str
        Title of the plot.
    max_points : int
        Maximum number of points to display.
    """
    if y_true.shape != y_pred.shape:
        raise ValueError('Shapes of y_true and y_pred must match for plotting.')
    if max_points <= 0:
        raise ValueError('max_points must be positive.')
    length = min(len(y_true), max_points)
    index = np.arange(length)
    plt.figure(figsize=(10, 4))
    plt.plot(index, y_true[:length], label='y_true')
    plt.plot(index, y_pred[:length], label='y_pred')
    plt.xlabel('time index')
    plt.ylabel('value')
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def build_informer_base_extra_args(device_type: str) -> List[str] | None:
    """
    Build base extra arguments list for Informer CLI based on device type string.

    Parameters
    ----------
    device_type : str
        Device type value from configuration or environment. Supported values:
        "auto", "cpu", "cuda", "mps".

    Returns
    -------
    list[str] | None
        List of extra CLI arguments for Informer or None if no extra arguments
        should be added (for example, when device_type == "auto").
    """
    if device_type == '':
        raise ValueError('device_type must not be empty.')
    normalized = device_type.strip().lower()
    if normalized == 'auto':
        return None
    if normalized in ('cpu', 'cuda', 'mps'):
        return ['--device_type', normalized]
    raise ValueError(
            f'Unsupported DEVICE_TYPE value "{device_type}". '
            f'Expected one of ["auto", "cpu", "cuda", "mps"].'
    )


class InformerOptunaExperimentRunner:
    """Manage Informer + Optuna experiments for multiple datasets and max_rows values."""

    def __init__(
            self,
            dataset_configs: List[DatasetConfig],
            experiment_config: InformerOptunaExperimentConfig,
    ) -> None:
        if not dataset_configs:
            raise ValueError('dataset_configs must not be empty.')
        if experiment_config.device_type == '':
            raise ValueError('experiment_config.device_type must not be empty.')
        self._dataset_configs = list(dataset_configs)
        self._experiment_config = experiment_config

    def run_all(self) -> Dict[InformerRunKey, ExperimentResult]:
        """Run experiments for all datasets and all max_rows values."""
        results: Dict[InformerRunKey, ExperimentResult] = {}
        for dataset_config in self._dataset_configs:
            for max_rows in self._experiment_config.max_rows_values:
                run_key = InformerRunKey(dataset_name=dataset_config.name, max_rows=max_rows)
                LOGGER.info(
                        f'Starting Informer+Optuna run for dataset="{run_key.dataset_name}", '
                        f'max_rows={run_key.max_rows}.'
                )
                result = self._run_single(dataset_config, max_rows)
                results[run_key] = result
                mse_value = float(result.metrics.get('mse', float('nan')))
                LOGGER.info(
                        f'Finished Informer+Optuna run for dataset="{run_key.dataset_name}", '
                        f'max_rows={run_key.max_rows}, mse={mse_value}.'
                )
        return results

    def _run_single(
            self,
            dataset_config: DatasetConfig,
            max_rows: int,
    ) -> ExperimentResult:
        """Run a single Informer + Optuna experiment for a dataset and max_rows."""
        if max_rows <= 0:
            raise ValueError('max_rows must be a positive integer.')
        if not dataset_config.csv_path.is_file():
            raise FileNotFoundError(
                    f'Dataset "{dataset_config.name}" CSV not found at "{dataset_config.csv_path}".'
            )

        trimmed_csv = prepare_trimmed_csv(
                original_csv_path=dataset_config.csv_path,
                dataset_name=dataset_config.name,
                max_rows=max_rows,
                datasets_root=INFORMER_DATASETS_ROOT,
        )

        metrics_root_for_dataset = self._experiment_config.metrics_root_dir / dataset_config.name / f'max_rows_{max_rows}'
        metrics_root_for_dataset.mkdir(parents=True, exist_ok=True)

        base_extra_args = build_informer_base_extra_args(
                device_type=self._experiment_config.device_type,
        )

        result = run_informer_optuna_etth_experiment(
                dataset_name=dataset_config.name,
                csv_path=str(trimmed_csv),
                informer_script_path=str(SRC_DIR / 'Informer2020' / 'informer_experiment_wrapper.py'),
                metrics_root_dir=str(metrics_root_for_dataset),
                base_extra_args=base_extra_args,
                timeout_seconds=self._experiment_config.timeout_seconds,
                model_name=f'informer-optuna-{dataset_config.name.lower()}',
                n_trials=self._experiment_config.n_trials,
        )
        return result

    @staticmethod
    def build_metrics_dataframe(
            results: Dict[InformerRunKey, ExperimentResult],
            primary_metric: str,
    ) -> pd.DataFrame:
        """Convert experiment results into a flat metrics DataFrame."""
        rows: List[Dict[str, float | int | str]] = []
        for run_key, result in results.items():
            metrics = result.metrics
            row: Dict[str, float | int | str] = {
                'dataset': run_key.dataset_name,
                'max_rows': run_key.max_rows,
            }
            for metric_name, metric_value in metrics.items():
                row[metric_name] = float(metric_value)
            if primary_metric not in row:
                row[primary_metric] = float('nan')
            rows.append(row)
        if not rows:
            return pd.DataFrame()
        df = pd.DataFrame(rows)
        df.sort_values(by=['dataset', 'max_rows'], inplace=True)
        df.reset_index(drop=True, inplace=True)
        return df

## Запуск экспериментов Informer + Optuna

В этом разделе:

1. Создаются конфигурации датасетов и эксперимента.
2. Запускаются все эксперименты Informer + Optuna для комбинаций датасет × `MAX_ROWS`.
3. Формируется таблица метрик по всем запускам.

In [ ]:
dataset_configs: List[DatasetConfig] = [
    DatasetConfig(name='ETTh1', csv_path=ETTH1_PATH, train_ratio=TRAIN_RATIO),
    DatasetConfig(name='ETTh2', csv_path=ETTH2_PATH, train_ratio=TRAIN_RATIO),
]

informer_experiment_config = InformerOptunaExperimentConfig(
        timeout_seconds=INFORMER_TIMEOUT_SECONDS,
        n_trials=INFORMER_N_TRIALS,
        metrics_root_dir=INFORMER_METRICS_ROOT,
        device_type=DEVICE_TYPE,
        max_rows_values=MAX_ROWS_VALUES,
        plots_max_points=PLOTS_MAX_POINTS,
)

informer_runner = InformerOptunaExperimentRunner(
        dataset_configs=dataset_configs,
        experiment_config=informer_experiment_config,
)

informer_results_by_run_key: Dict[InformerRunKey, ExperimentResult] = informer_runner.run_all()

primary_metric_name: str = 'mse'
informer_metrics_df = InformerOptunaExperimentRunner.build_metrics_dataframe(
        results=informer_results_by_run_key,
        primary_metric=primary_metric_name,
)
informer_metrics_df

## Визуализация результатов Informer

На этом шаге для каждой комбинации датасет × `MAX_ROWS`:

1. Строится график фактических значений `TARGET_COLUMN` на train-части выборки.
2. Загружается CSV с предсказаниями Informer и строится график фактических и предсказанных значений
   для отложенной выборки (валидация/тест).

In [ ]:
for run_key, result in informer_results_by_run_key.items():
    dataset_name = run_key.dataset_name
    max_rows = run_key.max_rows

    dataset_cfg = next((cfg for cfg in dataset_configs if cfg.name == dataset_name), None)
    if dataset_cfg is None:
        LOGGER.info(
                f'Visualization skipped for dataset="{dataset_name}", '
                f'max_rows={max_rows}: dataset config not found.'
        )
        continue

    trimmed_csv_path = (
            INFORMER_DATASETS_ROOT / dataset_name / f'{dataset_name}_max_rows_{max_rows}.csv'
    )
    if not trimmed_csv_path.is_file():
        LOGGER.info(
                f'Visualization skipped for dataset="{dataset_name}", '
                f'max_rows={max_rows}: trimmed CSV "{trimmed_csv_path}" not found.'
        )
        continue

    train_df, valid_df = load_train_valid_for_visualization(
            csv_path=trimmed_csv_path,
            max_rows=max_rows,
            train_ratio=dataset_cfg.train_ratio,
    )

    if TARGET_COLUMN not in train_df.columns:
        LOGGER.info(
                f'Visualization skipped for dataset="{dataset_name}", '
                f'max_rows={max_rows}: target column "{TARGET_COLUMN}" missing in train_df.'
        )
        continue

    plot_time_series_single(
            y_values=train_df[TARGET_COLUMN].to_numpy(),
            title=(
                f'Informer input train series '
                f'(dataset={dataset_name}, max_rows={max_rows}, column={TARGET_COLUMN})'
            ),
            max_points=PLOTS_MAX_POINTS,
            label='y_train',
    )

    predictions_csv_path_value = result.extra_info.get('predictions_csv_path')
    if not isinstance(predictions_csv_path_value, str):
        LOGGER.info(
                f'Informer predictions CSV path is missing for dataset="{dataset_name}", '
                f'max_rows={max_rows}; skipping prediction plot.'
        )
        continue

    predictions_csv_path = Path(predictions_csv_path_value)
    if not predictions_csv_path.is_file():
        LOGGER.info(
                f'Informer predictions CSV not found at "{predictions_csv_path}" for '
                f'dataset="{dataset_name}", max_rows={max_rows}; skipping prediction plot.'
        )
        continue

    df_pred = pd.read_csv(predictions_csv_path)
    if 'y_true' not in df_pred.columns or 'y_pred' not in df_pred.columns:
        LOGGER.info(
                f'Predictions CSV at "{predictions_csv_path}" does not contain '
                f'columns "y_true" and "y_pred"; skipping prediction plot.'
        )
        continue

    y_true = df_pred['y_true'].to_numpy()
    y_pred = df_pred['y_pred'].to_numpy()

    plot_true_vs_pred(
            y_true=y_true,
            y_pred=y_pred,
            title=(
                f'Informer+Optuna evaluation predictions '
                f'(dataset={dataset_name}, max_rows={max_rows})'
            ),
            max_points=PLOTS_MAX_POINTS,
    )